In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta  # Asegúrate de importar timedelta

# Visitas Tecnicas Brutas

In [2]:
ruta = r"C:\Users\wduran\OneDrive - CLAROCHILE\Digital Team\Triage Documentos Operaciones\Gestion Traige_wsp_masivo_VTR.csv"

tr = pd.read_csv(ruta, sep=";", encoding="utf-8")

In [3]:
tr = tr[tr["Periodo"].astype(str).str[:4] == "2026"]

In [4]:
tr.sample(1)

,MARCA,Nº DE SS,ÁREA,TIPO,RUT DEL CLIENTE,ID_VIVIENDA,CNC,FECHA DE CREACIÓN,ESTADO,Periodo
655847,VTR,1-272533055287,Reparación Técnica,Internet FTTH,9778010-2,5857041.0,CHIL029001,"2026-01-08 12:03:21,000",Anulada,2026-01


In [5]:
conteo_periodo = (
    tr.groupby("Periodo")
      .size()
      .reset_index(name="CANTIDAD")
      .sort_values("Periodo")
)

print("CONTEO POR PERIODO")
print(conteo_periodo)

CONTEO POR PERIODO
   Periodo  CANTIDAD
0  2026-01     62511
1  2026-02     59333
2  2026-03     19316


In [6]:
# Convertir FECHA DE CREACIÓN a datetime
tr["FECHA DE CREACIÓN"] = pd.to_datetime(tr["FECHA DE CREACIÓN"], errors="coerce")

# Crear campo SEMANA con semana ISO
tr["SEMANA"] = tr["FECHA DE CREACIÓN"].dt.isocalendar().week

C:\Users\wduran\AppData\Local\Temp\ipykernel_15312\2576618897.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tr["FECHA DE CREACIÓN"] = pd.to_datetime(tr["FECHA DE CREACIÓN"], errors="coerce")


In [7]:
# Convertir fecha
tr["FECHA DE CREACIÓN"] = pd.to_datetime(tr["FECHA DE CREACIÓN"], errors="coerce")

# Crear columnas ISO
iso = tr["FECHA DE CREACIÓN"].dt.isocalendar()
tr["ISO_YEAR"] = iso.year
tr["ISO_WEEK"] = iso.week

tr["YEAR_WEEK"] = (
    tr["ISO_YEAR"].astype(str)
    + "-W"
    + tr["ISO_WEEK"].astype(str).str.zfill(2)
)

In [8]:
# Obtener semanas únicas ordenadas
semanas = sorted(tr["YEAR_WEEK"].dropna().unique())

# Tomar últimas 4
ultimas_4 = semanas[-5:]

In [9]:
tr_4s = tr[tr["YEAR_WEEK"].isin(ultimas_4)]

In [10]:
conteo_4s = (
    tr_4s.groupby("YEAR_WEEK")
         .size()
         .reset_index(name="CANTIDAD")
         .sort_values("YEAR_WEEK")
)

print("\nCONTEO ÚLTIMAS 4 SEMANAS")
print(conteo_4s)


CONTEO ÚLTIMAS 4 SEMANAS
  YEAR_WEEK  CANTIDAD
0  2026-W07     15082
1  2026-W08     14335
2  2026-W09     14264
3  2026-W10     13695
4  2026-W11      4889


In [11]:
# Asegurar formato datetime
tr_4s["FECHA DE CREACIÓN"] = pd.to_datetime(tr_4s["FECHA DE CREACIÓN"])

# Filtrar solo el periodo 2026-03
periodo_2026_03 = tr_4s[tr_4s["Periodo"] == "2026-03"]

# Agrupar solo por día (sin hora)
conteo_dias = (
    periodo_2026_03
        .groupby(periodo_2026_03["FECHA DE CREACIÓN"].dt.date)
        .size()
        .reset_index(name="CANTIDAD")
        .sort_values("FECHA DE CREACIÓN")
)

print("\nCONTEO DIARIO - PERIODO 2026-03")
print(conteo_dias)


CONTEO DIARIO - PERIODO 2026-03
  FECHA DE CREACIÓN  CANTIDAD
0        2026-03-01       732
1        2026-03-02      2342
2        2026-03-03      2470
3        2026-03-04      2304
4        2026-03-05      2119
5        2026-03-06      2234
6        2026-03-07      1482
7        2026-03-08       744
8        2026-03-09      2453
9        2026-03-10      2436


C:\Users\wduran\AppData\Local\Temp\ipykernel_15312\2960282337.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tr_4s["FECHA DE CREACIÓN"] = pd.to_datetime(tr_4s["FECHA DE CREACIÓN"])


# Bases Asignaciones 

In [12]:
ruta = r"C:\Users\wduran\OneDrive - CLAROCHILE\Digital Team\Triage Documentos Operaciones\Asignacion_CLARO.csv"

ASIGCLARO = pd.read_csv(ruta, sep=";", encoding="utf-8")

In [13]:
ruta = r"C:\Users\wduran\OneDrive - CLAROCHILE\Digital Team\Triage Documentos Operaciones\Asignacion_VTR.csv"

AGIGVTR = pd.read_csv(ruta, sep=";", encoding="utf-8")

In [14]:
# Lista de palabras clave
keywords_casillero = r"CJUG|NJUG|EDES|NJUGCALL"

for df_base in [ASIGCLARO, AGIGVTR]:
    
    df_base["casillero"] = df_base["casillero"].astype(str)
    
    # Crear LOGICA_MASIVO
    df_base["LOGICA_MASIVO"] = (
        df_base["casillero"]
        .str.upper()
        .str.contains(keywords_casillero, na=False)
        .astype(int)
    )
    
    # Crear LOGICA (inversa de masivo)
    df_base["LOGICA"] = (df_base["LOGICA_MASIVO"] == 0).astype(int)

In [15]:
for df_base in [ASIGCLARO, AGIGVTR]:
    
    # Asegurar tipo texto
    df_base["Tipo_Asignacion"] = df_base["Tipo_Asignacion"].astype(str)
    
    # Crear campo Masivo
    df_base["Masivo"] = (
        df_base["Tipo_Asignacion"]
        .str.contains("Masivo", case=False, na=False)
        .astype(int)
    )
    
    # Crear campo LOGICA (inverso)
    df_base["LOGICA"] = (1 - df_base["Masivo"]).astype(int)

In [16]:
AGIGVTR.sample(1)

,estado,ID_SS,numero_solicitud,cuidad,fecha_agendamiento,hora,casillero,puesto_de_trabajo,rut_cliente,CNC,fecha,Periodo,Flag_INT,Llave,Semana,Tipo_Asignacion,LOGICA_MASIVO,LOGICA,Masivo
314516,PENDIENTE,1-3E2N4YF6,1-265727239746,ARICA,"2025-08-29 00:00:00,000",10:00:00-13:00:00,CJUG,Triage Internet,26984871-5,ARIC043001,"2025-08-28 14:41:57,000",2025-08,0,26984871-5_ARIC043001,S35,Masivo_Finalizado,1,0,1


In [17]:
# Convertir a datetime
ASIGCLARO["fecha"] = pd.to_datetime(ASIGCLARO["fecha"], errors="coerce")
AGIGVTR["fecha"] = pd.to_datetime(AGIGVTR["fecha"], errors="coerce")

C:\Users\wduran\AppData\Local\Temp\ipykernel_15312\458313721.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ASIGCLARO["fecha"] = pd.to_datetime(ASIGCLARO["fecha"], errors="coerce")
C:\Users\wduran\AppData\Local\Temp\ipykernel_15312\458313721.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  AGIGVTR["fecha"] = pd.to_datetime(AGIGVTR["fecha"], errors="coerce")


In [18]:
import pandas as pd

# Fecha de hoy
hoy = pd.Timestamp.today().normalize()

# Excluir registros del día actual
ASIGCLARO = ASIGCLARO[ASIGCLARO["fecha"] < hoy]
AGIGVTR = AGIGVTR[AGIGVTR["fecha"] < hoy]

# Crear columna MES (formato YYYY-MM)
ASIGCLARO["MES"] = ASIGCLARO["fecha"].dt.to_period("M")
AGIGVTR["MES"] = AGIGVTR["fecha"].dt.to_period("M")

# Conteo mensual
claro_mes = ASIGCLARO.groupby("MES").size()
vtr_mes = AGIGVTR.groupby("MES").size()

# Unir y sumar
conteo_mes = pd.concat([claro_mes, vtr_mes], axis=1)
conteo_mes.columns = ["CLARO", "VTR"]
conteo_mes = conteo_mes.fillna(0)

conteo_mes["TOTAL"] = conteo_mes["CLARO"] + conteo_mes["VTR"]

print("CONTEO MENSUAL")
print(conteo_mes.sort_index())

CONTEO MENSUAL
         CLARO    VTR  TOTAL
MES                         
2025-01   9265  31084  40349
2025-02   6697  29523  36220
2025-03   8130  35622  43752
2025-04   6424  31242  37666
2025-05   6219  26954  33173
2025-06   5578  27141  32719
2025-07   5599  26075  31674
2025-08   7452  26061  33513
2025-09   7802  24872  32674
2025-10   9910  31522  41432
2025-11   8877  33759  42636
2025-12   9263  29568  38831
2026-01   9252  31195  40447
2026-02   8612  29404  38016
2026-03   2779   8564  11343


C:\Users\wduran\AppData\Local\Temp\ipykernel_15312\622527548.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  AGIGVTR["MES"] = AGIGVTR["fecha"].dt.to_period("M")


In [19]:
import pandas as pd

# Asegurar formato fecha
ASIGCLARO["fecha"] = pd.to_datetime(ASIGCLARO["fecha"], errors="coerce")
AGIGVTR["fecha"] = pd.to_datetime(AGIGVTR["fecha"], errors="coerce")

# Crear año y semana ISO
for df_base in [ASIGCLARO, AGIGVTR]:
    iso = df_base["fecha"].dt.isocalendar()
    df_base["ISO_YEAR"] = iso.year
    df_base["ISO_WEEK"] = iso.week
    df_base["YEAR_WEEK"] = df_base["ISO_YEAR"].astype(str) + "-W" + df_base["ISO_WEEK"].astype(str).str.zfill(2)

C:\Users\wduran\AppData\Local\Temp\ipykernel_15312\1045156568.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  AGIGVTR["fecha"] = pd.to_datetime(AGIGVTR["fecha"], errors="coerce")


In [20]:
# Unimos semanas de ambas bases
semanas_disponibles = pd.concat([
    ASIGCLARO["YEAR_WEEK"],
    AGIGVTR["YEAR_WEEK"]
]).dropna().unique()

# Ordenar correctamente
semanas_disponibles = sorted(semanas_disponibles)

# Tomar últimas 3
ultimas_4 = semanas_disponibles[-5:]

In [21]:
claro_4s = ASIGCLARO[ASIGCLARO["YEAR_WEEK"].isin(ultimas_4)]
vtr_4s = AGIGVTR[AGIGVTR["YEAR_WEEK"].isin(ultimas_4)]

In [22]:
claro_sem = claro_4s.groupby("YEAR_WEEK").size()
vtr_sem = vtr_4s.groupby("YEAR_WEEK").size()

conteo_4s = pd.concat([claro_sem, vtr_sem], axis=1)
conteo_4s.columns = ["CLARO", "VTR"]
conteo_4s = conteo_4s.fillna(0)

conteo_4s["TOTAL"] = conteo_4s["CLARO"] + conteo_4s["VTR"]

print("\nCONTEO ÚLTIMAS 3 SEMANAS")
print(conteo_4s.sort_index())


CONTEO ÚLTIMAS 3 SEMANAS
           CLARO   VTR  TOTAL
YEAR_WEEK                    
2026-W07    2244  7546   9790
2026-W08    2041  6631   8672
2026-W09    1939  6400   8339
2026-W10    1976  6131   8107
2026-W11     649  2048   2697


In [23]:
# Asegurar marca
ASIGCLARO["MARCA"] = "CLARO"
AGIGVTR["MARCA"] = "VTR"

base_asig = pd.concat([ASIGCLARO, AGIGVTR], ignore_index=True)

# Resumen base por Periodo y Marca
resumen = (
    base_asig
    .groupby(["Periodo", "MARCA"])
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# Calcular totales por Periodo (CLARO + VTR)
totales_periodo = (
    resumen
    .groupby("Periodo")
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# Hacer merge para agregar los totales a cada fila
resumen_final = resumen.merge(totales_periodo, on="Periodo", how="left")

# Ordenar
resumen_final = resumen_final.sort_values(["Periodo", "MARCA"])

print(resumen_final)

    Periodo  MARCA  MASIVAS  LOGICAS  TOTAL  TOTAL_MASIVO  TOTAL_LOGICA
0   2025-01  CLARO     1781     7484   9265          9369         30980
1   2025-01    VTR     7588    23496  31084          9369         30980
2   2025-02  CLARO      617     6080   6697          5331         30889
3   2025-02    VTR     4714    24809  29523          5331         30889
4   2025-03  CLARO     1776     6354   8130         11369         32383
5   2025-03    VTR     9593    26029  35622         11369         32383
6   2025-04  CLARO     1605     4819   6424          9370         28296
7   2025-04    VTR     7765    23477  31242          9370         28296
8   2025-05  CLARO     1743     4476   6219          8498         24675
9   2025-05    VTR     6755    20199  26954          8498         24675
10  2025-06  CLARO     1694     3884   5578          9920         22799
11  2025-06    VTR     8226    18915  27141          9920         22799
12  2025-07  CLARO      796     4803   5599          7586       

In [24]:
# Asegurar marca
ASIGCLARO["MARCA"] = "CLARO"
AGIGVTR["MARCA"] = "VTR"

base_asig = pd.concat([ASIGCLARO, AGIGVTR], ignore_index=True)

# Resumen base por Periodo y Marca
resumen = (
    base_asig
    .groupby(["YEAR_WEEK", "MARCA"])
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# Calcular totales por Periodo (CLARO + VTR)
totales_periodo = (
    resumen
    .groupby("YEAR_WEEK")
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# Hacer merge para agregar los totales a cada fila
resumen_final = resumen.merge(totales_periodo, on="YEAR_WEEK", how="left")

# Ordenar
resumen_final = resumen_final.sort_values(["YEAR_WEEK", "MARCA"])

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print(resumen_final)

    YEAR_WEEK  MARCA  MASIVAS  LOGICAS  TOTAL  TOTAL_MASIVO  TOTAL_LOGICA
0    2025-W01  CLARO       77      958   1035           658          3798
1    2025-W01    VTR      581     2840   3421           658          3798
2    2025-W02  CLARO      374     1885   2259          2310          7280
3    2025-W02    VTR     1936     5395   7331          2310          7280
4    2025-W03  CLARO      553     1617   2170          2329          6996
5    2025-W03    VTR     1776     5379   7155          2329          6996
6    2025-W04  CLARO      422     1655   2077          2091          7132
7    2025-W04    VTR     1669     5477   7146          2091          7132
8    2025-W05  CLARO      372     1721   2093          2343          7061
9    2025-W05    VTR     1971     5340   7311          2343          7061
10   2025-W06  CLARO      380     1511   1891          2208          7335
11   2025-W06    VTR     1828     5824   7652          2208          7335
12   2025-W07  CLARO       98     1369

In [25]:
import pandas as pd

# ==============================
# ASEGURAR MARCA
# ==============================
ASIGCLARO["MARCA"] = "CLARO"
AGIGVTR["MARCA"] = "VTR"

# ==============================
# UNIR BASES
# ==============================
base_asig = pd.concat([ASIGCLARO, AGIGVTR], ignore_index=True)

# ==============================
# ASEGURAR FECHA EN FORMATO DATETIME
# ==============================
base_asig["fecha"] = pd.to_datetime(base_asig["fecha"])

# ==============================
# FILTRAR PERIODO 2026-03
# ==============================
base_2026_03 = base_asig[base_asig["Periodo"] == "2026-03"].copy()

# Crear columna solo fecha (sin hora)
base_2026_03["FECHA_DIA"] = base_2026_03["fecha"].dt.date

# ==============================
# RESUMEN POR DIA Y MARCA
# ==============================
resumen = (
    base_2026_03
    .groupby(["FECHA_DIA", "MARCA"])
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# ==============================
# TOTALES POR DIA (CLARO + VTR)
# ==============================
totales_dia = (
    resumen
    .groupby("FECHA_DIA")
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# ==============================
# MERGE TOTALES
# ==============================
resumen_final = resumen.merge(
    totales_dia,
    on="FECHA_DIA",
    how="left"
)

# ==============================
# ORDENAR
# ==============================
resumen_final = resumen_final.sort_values(["FECHA_DIA", "MARCA"])

print("\n===== RESUMEN DIARIO ASIGNACIONES - PERIODO 2026-03 =====")
print(resumen_final)


===== RESUMEN DIARIO ASIGNACIONES - PERIODO 2026-03 =====
     FECHA_DIA  MARCA  MASIVAS  LOGICAS  TOTAL  TOTAL_MASIVO  TOTAL_LOGICA
0   2026-03-01  CLARO        2      153    154            49           493
1   2026-03-01    VTR       47      340    385            49           493
2   2026-03-02  CLARO       11      205    216            99          1057
3   2026-03-02    VTR       88      852    938            99          1057
4   2026-03-03  CLARO       14      395    408           121          1382
5   2026-03-03    VTR      107      987   1091           121          1382
6   2026-03-04  CLARO       15      355    367           134          1199
7   2026-03-04    VTR      119      844    955           134          1199
8   2026-03-05  CLARO       14      330    341           154          1185
9   2026-03-05    VTR      140      855    988           154          1185
10  2026-03-06  CLARO       11      214    223           171          1131
11  2026-03-06    VTR      160      917  

# Bases Recorrido

In [26]:
ruta = r"C:\Users\wduran\OneDrive - CLAROCHILE\Digital Team\Triage Documentos Operaciones\Recorrido_VTR.csv"

recoVTR = pd.read_csv(ruta, sep=";", encoding="utf-8")

In [27]:
ruta = r"C:\Users\wduran\OneDrive - CLAROCHILE\Digital Team\Triage Documentos Operaciones\Recorrido_CLARO.csv"

recoCLARO = pd.read_csv(ruta, sep=";", encoding="utf-8")

In [28]:
# Lista de palabras clave
keywords_casillero = r"CJUG|NJUG|EDES|NJUGCALL"

for df_base in [recoVTR]:
    
    df_base["CASILLERO"] = df_base["CASILLERO"].astype(str)
    
    # Crear LOGICA_MASIVO
    df_base["LOGICA_MASIVO"] = (
        df_base["CASILLERO"]
        .str.upper()
        .str.contains(keywords_casillero, na=False)
        .astype(int)
    )
    
    # Crear LOGICA (inversa de masivo)
    df_base["LOGICA"] = (df_base["LOGICA_MASIVO"] == 0).astype(int)

In [29]:
recoVTR["LOGICA"].value_counts()

LOGICA
1    339091
0    100930
Name: count, dtype: int64

In [30]:
for df_base in [recoCLARO]:
    
    df_base["PUESTO_TRABAJO"] = df_base["PUESTO_TRABAJO"].astype(str)
    
    # Crear LOGICA_MASIVO
    df_base["LOGICA_MASIVO"] = (
        df_base["PUESTO_TRABAJO"]
        .str.contains("Masivo", case=False, na=False)
        .astype(int)
    )
    
    # Crear LOGICA (inversa)
    df_base["LOGICA"] = (df_base["LOGICA_MASIVO"] == 0).astype(int)

In [31]:
recoCLARO["LOGICA"].value_counts()

LOGICA
1    108185
0     33396
Name: count, dtype: int64

In [32]:
# Convertir a datetime
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")

# Extraer YYYY-MM-DD
recoVTR["FECHA_INICIO"] = recoVTR["FECHA_INICIO"].dt.date

In [33]:
# Convertir a datetime
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# Extraer YYYY-MM-DD
recoCLARO["FECHA_INICIO"] = recoCLARO["FECHA_INICIO"].dt.date

In [34]:
# Convertir FECHA_INICIO a datetime
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")

# Crear campo SEMANA con semana ISO
recoVTR["SEMANA"] = recoVTR["FECHA_INICIO"].dt.isocalendar().week

In [35]:
# Convertir FECHA_INICIO a datetime
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# Crear campo SEMANA con semana ISO
recoCLARO["SEMANA"] = recoCLARO["FECHA_INICIO"].dt.isocalendar().week

In [36]:
# Ajustar las opciones para mostrar todas las columnas
pd.set_option('display.max_columns', None)

In [37]:
recoVTR = recoVTR[
    ~(
        recoVTR["GESTION"].str.contains(
            "Orden de reparacion ya cancelada",
            case=False,
            na=False
        )
        |
        recoVTR["CASILLERO"].str.contains(
            "CP05",
            case=False,
            na=False
        )
    )
]

In [38]:
recoCLARO = recoCLARO[
    ~(
        recoCLARO["GESTION"].str.contains(
            "no corresponde gesti",
            case=False,
            na=False
        ))
]

In [39]:
# Convertir a fecha
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# Eliminar registros sin identificador o fecha
recoVTR = recoVTR.dropna(subset=["NMRO_ORDEN", "FECHA_INICIO"])
recoCLARO = recoCLARO.dropna(subset=["FOLIO", "FECHA_INICIO"])

# Ordenar por fecha
recoVTR = recoVTR.sort_values("FECHA_INICIO")
recoCLARO = recoCLARO.sort_values("FECHA_INICIO")

# Tomar el último registro por orden/folio
recoVTR = recoVTR.drop_duplicates(subset="NMRO_ORDEN", keep="last")
recoCLARO = recoCLARO.drop_duplicates(subset="FOLIO", keep="last")

# Conteo por periodo
vtr_periodo = recoVTR["Periodo"].value_counts()
claro_periodo = recoCLARO["Periodo"].value_counts()

# Unir resultados
conteo_periodo = pd.concat([vtr_periodo, claro_periodo], axis=1)
conteo_periodo.columns = ["VTR", "CLARO"]
conteo_periodo = conteo_periodo.fillna(0)

# Total
conteo_periodo["TOTAL"] = conteo_periodo["VTR"] + conteo_periodo["CLARO"]

print("\nCONTEO RECORRIDO")
print(conteo_periodo.sort_index())

C:\Users\wduran\AppData\Local\Temp\ipykernel_15312\2495549729.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")



CONTEO RECORRIDO
           VTR  CLARO  TOTAL
Periodo                     
2025-01  26052   8999  35051
2025-02  25082   7481  32563
2025-03  29773   8010  37783
2025-04  27783   7490  35273
2025-05  24655   7559  32214
2025-06  23463   7535  30998
2025-07  22887   7623  30510
2025-08  22720   8086  30806
2025-09  21989   7810  29799
2025-10  27252   9916  37168
2025-11  29310   8413  37723
2025-12  26738   9247  35985
2026-01  28150   9147  37297
2026-02  27345   9207  36552
2026-03   8877   3451  12328


In [40]:
# Convertir a fecha
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# Eliminar registros sin identificador o fecha
recoVTR = recoVTR.dropna(subset=["NMRO_ORDEN", "FECHA_INICIO"])
recoCLARO = recoCLARO.dropna(subset=["FOLIO", "FECHA_INICIO"])

# Ordenar por fecha
recoVTR = recoVTR.sort_values("FECHA_INICIO")
recoCLARO = recoCLARO.sort_values("FECHA_INICIO")

# Quedarse con el último registro por orden/folio
recoVTR = recoVTR.drop_duplicates(subset="NMRO_ORDEN", keep="last")
recoCLARO = recoCLARO.drop_duplicates(subset="FOLIO", keep="last")

# Agregar marca
recoCLARO["MARCA"] = "CLARO"
recoVTR["MARCA"] = "VTR"

# Unir bases
base_reco = pd.concat([recoCLARO, recoVTR], ignore_index=True)

# Resumen por Periodo y Marca
resumen = (
    base_reco
    .groupby(["Periodo", "MARCA"], observed=True)
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# Totales por Periodo
totales_periodo = (
    resumen
    .groupby("Periodo", observed=True)
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# Agregar totales
resumen_final = resumen.merge(
    totales_periodo,
    on="Periodo",
    how="left"
)

# Ordenar
resumen_final = resumen_final.sort_values(["Periodo", "MARCA"])

print("\nCONTEO RECORRIDO POR LOGICA")
print(resumen_final)


CONTEO RECORRIDO POR LOGICA
    Periodo  MARCA  MASIVAS  LOGICAS  TOTAL  TOTAL_MASIVO  TOTAL_LOGICA
0   2025-01  CLARO     2487     6512   8999          9984         25067
1   2025-01    VTR     7497    18555  26052          9984         25067
2   2025-02  CLARO     2737     4744   7481          8016         24547
3   2025-02    VTR     5279    19803  25082          8016         24547
4   2025-03  CLARO     3611     4399   8010         12826         24957
5   2025-03    VTR     9215    20558  29773         12826         24957
6   2025-04  CLARO     4088     3402   7490         12316         22957
7   2025-04    VTR     8228    19555  27783         12316         22957
8   2025-05  CLARO     3384     4175   7559         10649         21565
9   2025-05    VTR     7265    17390  24655         10649         21565
10  2025-06  CLARO     3442     4093   7535         11935         19063
11  2025-06    VTR     8493    14970  23463         11935         19063
12  2025-07  CLARO     2154     546

In [41]:
for df_base in [recoVTR, recoCLARO]:

    df_base["FECHA_INICIO"] = pd.to_datetime(
        df_base["FECHA_INICIO"],
        errors="coerce"
    )

    iso = df_base["FECHA_INICIO"].dt.isocalendar()

    df_base["ISO_YEAR"] = iso.year
    df_base["ISO_WEEK"] = iso.week

    # Crear YEAR_WEEK solo si no es nulo
    df_base["YEAR_WEEK"] = (
        df_base["ISO_YEAR"].astype("Int64").astype(str)
        + "-W"
        + df_base["ISO_WEEK"].astype("Int64").astype(str).str.zfill(2)
    )

In [42]:
# =========================
# LIMPIEZA Y ÚLTIMO REGISTRO
# =========================

# Convertir fecha
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# Eliminar registros sin identificador o fecha
recoVTR = recoVTR.dropna(subset=["NMRO_ORDEN", "FECHA_INICIO"])
recoCLARO = recoCLARO.dropna(subset=["FOLIO", "FECHA_INICIO"])

# Ordenar por fecha
recoVTR = recoVTR.sort_values("FECHA_INICIO")
recoCLARO = recoCLARO.sort_values("FECHA_INICIO")

# Quedarse con el último registro por orden/folio
recoVTR = recoVTR.drop_duplicates(subset="NMRO_ORDEN", keep="last")
recoCLARO = recoCLARO.drop_duplicates(subset="FOLIO", keep="last")

# =========================
# PREPARAR BASE
# =========================

recoCLARO["MARCA"] = "CLARO"
recoVTR["MARCA"] = "VTR"

# Unir bases
base_reco = pd.concat([recoCLARO, recoVTR], ignore_index=True)

# =========================
# RESUMEN POR SEMANA
# =========================

resumen = (
    base_reco
    .groupby(["YEAR_WEEK", "MARCA"], observed=True)
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# Totales por semana
totales_periodo = (
    resumen
    .groupby("YEAR_WEEK", observed=True)
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# Merge totales
resumen_final = resumen.merge(
    totales_periodo,
    on="YEAR_WEEK",
    how="left"
)

# Ordenar
resumen_final = resumen_final.sort_values(["YEAR_WEEK", "MARCA"])

# =========================
# 🔥 ÚLTIMAS SEMANAS
# =========================

semanas = (
    resumen_final["YEAR_WEEK"]
    .dropna()
    .sort_values()
    .unique()
)

ultimas_4 = semanas[-6:]

resumen_ultimas_4 = resumen_final[
    resumen_final["YEAR_WEEK"].isin(ultimas_4)
]

print("\n===== RECO - ÚLTIMAS SEMANAS =====")
print(resumen_ultimas_4)


===== RECO - ÚLTIMAS SEMANAS =====
    YEAR_WEEK  MARCA  MASIVAS  LOGICAS  TOTAL  TOTAL_MASIVO  TOTAL_LOGICA
114  2026-W06  CLARO      111     2329   2440          2526          8208
115  2026-W06    VTR     2415     5879   8294          2526          8208
116  2026-W07  CLARO       95     2279   2374          1692          7822
117  2026-W07    VTR     1597     5543   7140          1692          7822
118  2026-W08  CLARO       64     2277   2341           844          7761
119  2026-W08    VTR      780     5484   6264           844          7761
120  2026-W09  CLARO       59     1907   1966           766          6596
121  2026-W09    VTR      707     4689   5396           766          6596
122  2026-W10  CLARO       81     2406   2487           944          7782
123  2026-W10    VTR      863     5376   6239           944          7782
124  2026-W11  CLARO       20      772    792           308          2750
125  2026-W11    VTR      288     1978   2266           308          2750


In [43]:
for df_base in [recoVTR, recoCLARO]:

    df_base["FECHA_INICIO"] = pd.to_datetime(
        df_base["FECHA_INICIO"],
        errors="coerce"
    )

    iso = df_base["FECHA_INICIO"].dt.isocalendar()

    df_base["ISO_YEAR"] = iso.year
    df_base["ISO_WEEK"] = iso.week

    # Crear YEAR_WEEK solo si no es nulo
    df_base["YEAR_WEEK"] = (
        df_base["ISO_YEAR"].astype("Int64").astype(str)
        + "-W"
        + df_base["ISO_WEEK"].astype("Int64").astype(str).str.zfill(2)
    )

In [44]:
semanas = sorted(
    pd.concat([
        recoVTR["YEAR_WEEK"],
        recoCLARO["YEAR_WEEK"]
    ])
    .dropna()
    .loc[lambda x: ~x.str.contains("<NA>")]  # elimina basura
    .unique()
)

ultimas_4 = semanas[-20:]

In [45]:
# =========================
# LIMPIEZA Y ÚLTIMO REGISTRO
# =========================

# Convertir fecha
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# Eliminar registros sin identificador o fecha
recoVTR = recoVTR.dropna(subset=["NMRO_ORDEN", "FECHA_INICIO"])
recoCLARO = recoCLARO.dropna(subset=["FOLIO", "FECHA_INICIO"])

# Ordenar por fecha
recoVTR = recoVTR.sort_values("FECHA_INICIO")
recoCLARO = recoCLARO.sort_values("FECHA_INICIO")

# Quedarse con el último registro por orden/folio
recoVTR = recoVTR.drop_duplicates(subset="NMRO_ORDEN", keep="last")
recoCLARO = recoCLARO.drop_duplicates(subset="FOLIO", keep="last")

# =========================
# FILTRAR ÚLTIMAS SEMANAS
# =========================

vtr_4s = recoVTR[recoVTR["YEAR_WEEK"].isin(ultimas_4)]
claro_4s = recoCLARO[recoCLARO["YEAR_WEEK"].isin(ultimas_4)]

# Conteo individual
vtr_sem = vtr_4s.groupby("YEAR_WEEK").size()
claro_sem = claro_4s.groupby("YEAR_WEEK").size()

# Unir
conteo_sem = pd.concat([vtr_sem, claro_sem], axis=1)
conteo_sem.columns = ["VTR", "CLARO"]
conteo_sem = conteo_sem.fillna(0).astype(int)

# Total
conteo_sem["TOTAL"] = (
    conteo_sem["VTR"] + conteo_sem["CLARO"]
).astype(int)

print("\nCONTEO RECORRIDO ÚLTIMAS 4 SEMANAS")
print(conteo_sem.sort_index())


CONTEO RECORRIDO ÚLTIMAS 4 SEMANAS
            VTR  CLARO  TOTAL
YEAR_WEEK                    
2025-W44   5820   2206   8026
2025-W45   7895   2216  10111
2025-W46   7287   2028   9315
2025-W47   6713   1761   8474
2025-W48   6242   2026   8268
2025-W49   6589   2330   8919
2025-W50   7196   2482   9678
2025-W51   6015   1928   7943
2025-W52   4459   1713   6172
2026-W01   4451   1532   5983
2026-W02   5853   2020   7873
2026-W03   6846   2142   8988
2026-W04   6495   2104   8599
2026-W05   7607   2401  10008
2026-W06   8294   2440  10734
2026-W07   7140   2374   9514
2026-W08   6264   2341   8605
2026-W09   5396   1966   7362
2026-W10   6239   2487   8726
2026-W11   2266    792   3058


In [46]:
import pandas as pd

# =========================
# ASEGURAR FORMATO FECHA
# =========================
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# =========================
# ELIMINAR REGISTROS INVALIDOS
# =========================
recoVTR = recoVTR.dropna(subset=["NMRO_ORDEN", "FECHA_INICIO"])
recoCLARO = recoCLARO.dropna(subset=["FOLIO", "FECHA_INICIO"])

# =========================
# ORDENAR POR FECHA
# =========================
recoVTR = recoVTR.sort_values("FECHA_INICIO")
recoCLARO = recoCLARO.sort_values("FECHA_INICIO")

# =========================
# TOMAR ULTIMO REGISTRO POR ORDEN
# =========================
recoVTR = recoVTR.drop_duplicates(subset="NMRO_ORDEN", keep="last")
recoCLARO = recoCLARO.drop_duplicates(subset="FOLIO", keep="last")

# =========================
# ASEGURAR MARCA
# =========================
recoCLARO["MARCA"] = "CLARO"
recoVTR["MARCA"] = "VTR"

# =========================
# UNIR BASES
# =========================
base_reco = pd.concat([recoCLARO, recoVTR], ignore_index=True)

# =========================
# CREAR COLUMNA DIA
# =========================
base_reco["DIA"] = base_reco["FECHA_INICIO"].dt.date

# =========================
# FILTRAR SOLO PERIODO 2026-03
# =========================
base_reco = base_reco[base_reco["Periodo"] == "2026-03"]

# =========================
# RESUMEN POR DIA Y MARCA
# =========================
resumen = (
    base_reco
    .groupby(["DIA", "MARCA"], observed=True)
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# =========================
# TOTALES POR DIA
# =========================
totales_dia = (
    resumen
    .groupby("DIA", observed=True)
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# =========================
# MERGE TOTALES
# =========================
resumen_final = resumen.merge(
    totales_dia,
    on="DIA",
    how="left"
)

# =========================
# ORDENAR
# =========================
resumen_final = resumen_final.sort_values(["DIA", "MARCA"])

print("\nCONTEO RECORRIDO POR LOGICA - DIARIO PERIODO 2026-03")
print(resumen_final)


CONTEO RECORRIDO POR LOGICA - DIARIO PERIODO 2026-03
           DIA  MARCA  MASIVAS  LOGICAS  TOTAL  TOTAL_MASIVO  TOTAL_LOGICA
0   2026-03-01  CLARO        4      168    172            48           496
1   2026-03-01    VTR       44      328    372            48           496
2   2026-03-02  CLARO        6      340    346           142          1194
3   2026-03-02    VTR      136      854    990           142          1194
4   2026-03-03  CLARO       13      474    487           146          1342
5   2026-03-03    VTR      133      868   1001           146          1342
6   2026-03-04  CLARO       17      411    428           131          1407
7   2026-03-04    VTR      114      996   1110           131          1407
8   2026-03-05  CLARO       14      418    432           198          1254
9   2026-03-05    VTR      184      836   1020           198          1254
10  2026-03-06  CLARO       11      353    364           162          1269
11  2026-03-06    VTR      151      916   1067

# Bases Contactabilidad

In [47]:
# Eliminar registros que contengan "sin contacto"
recoVTR = recoVTR[
    ~recoVTR["GESTION"].str.contains("sin contacto", case=False, na=False)
].copy()

recoCLARO = recoCLARO[
    ~recoCLARO["GESTION"].str.contains("sin contacto", case=False, na=False)
].copy()

In [48]:
# Conteo individual
vtr_periodo = recoVTR.groupby("Periodo").size()
claro_periodo = recoCLARO.groupby("Periodo").size()

# Unir
conteo_periodo = pd.concat([vtr_periodo, claro_periodo], axis=1)
conteo_periodo.columns = ["VTR", "CLARO"]
conteo_periodo = conteo_periodo.fillna(0).astype(int)

# Total
conteo_periodo["TOTAL"] = (
    conteo_periodo["VTR"] + conteo_periodo["CLARO"]
).astype(int)

print("\n===== Contactabilidad POR PERIODO =====")
print(conteo_periodo.sort_index())


===== Contactabilidad POR PERIODO =====
           VTR  CLARO  TOTAL
Periodo                     
2025-01  15235   6526  21761
2025-02  14749   5730  20479
2025-03  18209   6091  24300
2025-04  18487   5088  23575
2025-05  16455   5253  21708
2025-06  19004   5897  24901
2025-07  17142   5517  22659
2025-08  14959   4866  19825
2025-09  13319   5008  18327
2025-10  18659   7102  25761
2025-11  21591   5206  26797
2025-12  19698   5154  24852
2026-01  20226   5235  25461
2026-02  20634   5409  26043
2026-03   6589   1971   8560


In [49]:
# Asegurar fecha
for df_base in [recoVTR, recoCLARO]:
    df_base["FECHA_INICIO"] = pd.to_datetime(
        df_base["FECHA_INICIO"],
        errors="coerce"
    )
    
    iso = df_base["FECHA_INICIO"].dt.isocalendar()
    df_base["ISO_YEAR"] = iso.year
    df_base["ISO_WEEK"] = iso.week
    
    df_base["YEAR_WEEK"] = (
        df_base["ISO_YEAR"].astype(str)
        + "-W"
        + df_base["ISO_WEEK"].astype(str).str.zfill(2)
    )

In [50]:
# Conteo individual
vtr_sem = recoVTR.groupby("YEAR_WEEK").size()
claro_sem = recoCLARO.groupby("YEAR_WEEK").size()

# Unir
conteo_semana = pd.concat([vtr_sem, claro_sem], axis=1)
conteo_semana.columns = ["VTR", "CLARO"]
conteo_semana = conteo_semana.fillna(0).astype(int)

# Total
conteo_semana["TOTAL"] = (
    conteo_semana["VTR"] + conteo_semana["CLARO"]
).astype(int)

print("\n===== Contactabilidad POR SEMANA =====")
print(conteo_semana.sort_index())


===== Contactabilidad POR SEMANA =====
            VTR  CLARO  TOTAL
YEAR_WEEK                    
2025-W01   1781    626   2407
2025-W02   3651   1470   5121
2025-W03   3569   1430   4999
2025-W04   3312   1576   4888
2025-W05   3604   1761   5365
2025-W06   3954   1487   5441
2025-W07   3778   1334   5112
2025-W08   3543   1310   4853
2025-W09   3453   1591   5044
2025-W10   3642   1670   5312
2025-W11   4104   1435   5539
2025-W12   4589   1483   6072
2025-W13   4378   1008   5386
2025-W14   4630   1158   5788
2025-W15   4520   1166   5686
2025-W16   3816   1078   4894
2025-W17   4342   1147   5489
2025-W18   3843   1180   5023
2025-W19   3776   1131   4907
2025-W20   3750   1487   5237
2025-W21   3708   1197   4905
2025-W22   3651   1020   4671
2025-W23   3699   1193   4892
2025-W24   4449   1281   5730
2025-W25   4978   1591   6569
2025-W26   4886   1527   6413
2025-W27   4332   1506   5838
2025-W28   3913   1124   5037
2025-W29   3680   1231   4911
2025-W30   3752   1305   5057


In [51]:
import pandas as pd

# ==============================
# ASEGURAR MARCA
# ==============================
recoCLARO["MARCA"] = "CLARO"
recoVTR["MARCA"] = "VTR"

# ==============================
# UNIR BASES
# ==============================
base_reco = pd.concat([recoCLARO, recoVTR], ignore_index=True)

# ==============================
# RESUMEN POR PERIODO Y MARCA
# ==============================
resumen = (
    base_reco
    .groupby(["Periodo", "MARCA"])
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# ==============================
# TOTALES POR PERIODO (CLARO + VTR)
# ==============================
totales_periodo = (
    resumen
    .groupby("Periodo")
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# ==============================
# MERGE TOTALES
# ==============================
resumen_final = resumen.merge(
    totales_periodo,
    on="Periodo",
    how="left"
)

# ==============================
# ORDENAR
# ==============================
resumen_final = resumen_final.sort_values(["Periodo", "MARCA"])

# ==============================
# PRINT FINAL
# ==============================
print("\n===== CONTACTABILIDAD - RESUMEN POR PERIODO =====")
print(resumen_final)


===== CONTACTABILIDAD - RESUMEN POR PERIODO =====
    Periodo  MARCA  MASIVAS  LOGICAS  TOTAL  TOTAL_MASIVO  TOTAL_LOGICA
0   2025-01  CLARO     2071     4455   6526          6628         15133
1   2025-01    VTR     4557    10678  15235          6628         15133
2   2025-02  CLARO     2396     3334   5730          5710         14769
3   2025-02    VTR     3314    11435  14749          5710         14769
4   2025-03  CLARO     2933     3158   6091          8789         15511
5   2025-03    VTR     5856    12353  18209          8789         15511
6   2025-04  CLARO     2824     2264   5088          8337         15238
7   2025-04    VTR     5513    12974  18487          8337         15238
8   2025-05  CLARO     2421     2832   5253          7327         14381
9   2025-05    VTR     4906    11549  16455          7327         14381
10  2025-06  CLARO     2772     3125   5897         10632         14269
11  2025-06    VTR     7860    11144  19004         10632         14269
12  2025-07  

In [52]:
# ==============================
# ASEGURAR MARCA
# ==============================
recoCLARO["MARCA"] = "CLARO"
recoVTR["MARCA"] = "VTR"

# ==============================
# UNIR BASES
# ==============================
base_reco = pd.concat([recoCLARO, recoVTR], ignore_index=True)

# ==============================
# RESUMEN POR SEMANA Y MARCA
# ==============================
resumen = (
    base_reco
    .groupby(["YEAR_WEEK", "MARCA"])
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# ==============================
# TOTALES POR SEMANA (CLARO + VTR)
# ==============================
totales_semana = (
    resumen
    .groupby("YEAR_WEEK")
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# ==============================
# MERGE TOTALES
# ==============================
resumen_final = resumen.merge(
    totales_semana,
    on="YEAR_WEEK",
    how="left"
)

# ==============================
# ORDENAR
# ==============================
resumen_final = resumen_final.sort_values(["YEAR_WEEK", "MARCA"])

# ==============================
# FILTRAR ULTIMAS 5 SEMANAS
# ==============================
ultimas_5 = sorted(resumen_final["YEAR_WEEK"].unique())[-6:]

resumen_final_ult5 = (
    resumen_final[
        resumen_final["YEAR_WEEK"].isin(ultimas_5)
    ]
    .sort_values(["YEAR_WEEK", "MARCA"])
)

# ==============================
# PRINT FINAL
# ==============================
print("\n===== CONTACTABILIDAD - RESUMEN ULTIMAS 5 SEMANAS =====")
print(resumen_final_ult5)


===== CONTACTABILIDAD - RESUMEN ULTIMAS 5 SEMANAS =====
    YEAR_WEEK  MARCA  MASIVAS  LOGICAS  TOTAL  TOTAL_MASIVO  TOTAL_LOGICA
114  2026-W06  CLARO        3     1448   1451          2354          5594
115  2026-W06    VTR     2351     4146   6497          2354          5594
116  2026-W07  CLARO        4     1456   1460          1477          5439
117  2026-W07    VTR     1473     3983   5456          1477          5439
118  2026-W08  CLARO       11     1326   1337           759          5145
119  2026-W08    VTR      748     3819   4567           759          5145
120  2026-W09  CLARO        4     1175   1179           663          4388
121  2026-W09    VTR      659     3213   3872           663          4388
122  2026-W10  CLARO       18     1358   1376           843          5290
123  2026-W10    VTR      825     3932   4757           843          5290
124  2026-W11  CLARO        0      491    491           267          1792
125  2026-W11    VTR      267     1301   1568          

In [53]:
# =========================
# ASEGURAR MARCA
# =========================
recoCLARO["MARCA"] = "CLARO"
recoVTR["MARCA"] = "VTR"

# =========================
# UNIR BASES
# =========================
base_reco = pd.concat([recoCLARO, recoVTR], ignore_index=True)

# =========================
# ASEGURAR FORMATO FECHA
# =========================
base_reco["FECHA_INICIO"] = pd.to_datetime(base_reco["FECHA_INICIO"])

# Crear columna DIA
base_reco["DIA"] = base_reco["FECHA_INICIO"].dt.date

# =========================
# FILTRAR SOLO PERIODO 2026-03
# =========================
base_reco = base_reco[base_reco["Periodo"] == "2026-03"]

# =========================
# RESUMEN POR DIA Y MARCA
# =========================
resumen = (
    base_reco
    .groupby(["DIA", "MARCA"])
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# =========================
# TOTALES POR DIA
# =========================
totales_dia = (
    resumen
    .groupby("DIA")
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# =========================
# MERGE TOTALES
# =========================
resumen_final = resumen.merge(
    totales_dia,
    on="DIA",
    how="left"
)

# =========================
# ORDENAR
# =========================
resumen_final = resumen_final.sort_values(["DIA", "MARCA"])

print("\nCONTEO CONTACTADO POR LOGICA - DIARIO PERIODO 2026-03")
print(resumen_final)


CONTEO CONTACTADO POR LOGICA - DIARIO PERIODO 2026-03
           DIA  MARCA  MASIVAS  LOGICAS  TOTAL  TOTAL_MASIVO  TOTAL_LOGICA
0   2026-03-01  CLARO        1      103    104            44           324
1   2026-03-01    VTR       43      221    264            44           324
2   2026-03-02  CLARO        0      184    184           129           811
3   2026-03-02    VTR      129      627    756           129           811
4   2026-03-03  CLARO        1      264    265           126           899
5   2026-03-03    VTR      125      635    760           126           899
6   2026-03-04  CLARO        6      225    231           113           980
7   2026-03-04    VTR      107      755    862           113           980
8   2026-03-05  CLARO        3      228    231           179           858
9   2026-03-05    VTR      176      630    806           179           858
10  2026-03-06  CLARO        3      212    215           147           856
11  2026-03-06    VTR      144      644    78

# Cancelaciones

In [54]:
recoVTR = recoVTR[
    recoVTR["Resultado Final Orden"]
        .str.replace(r"\s+", " ", regex=True)
        .str.contains("cancelado", case=False, na=False)
].copy()

recoCLARO = recoCLARO[
    recoCLARO["Resultado Final Orden"]
        .str.replace(r"\s+", " ", regex=True)
        .str.contains("cancelado", case=False, na=False)
].copy()

In [55]:
# Conteo individual
vtr_periodo = recoVTR.groupby("Periodo").size()
claro_periodo = recoCLARO.groupby("Periodo").size()

# Unir
conteo_periodo = pd.concat([vtr_periodo, claro_periodo], axis=1)
conteo_periodo.columns = ["VTR", "CLARO"]
conteo_periodo = conteo_periodo.fillna(0).astype(int)

# Total
conteo_periodo["TOTAL"] = conteo_periodo["VTR"] + conteo_periodo["CLARO"]

print("\n===== CANCELADOS - CONTEO POR PERIODO =====")
print(conteo_periodo.sort_index())


===== CANCELADOS - CONTEO POR PERIODO =====
           VTR  CLARO  TOTAL
Periodo                     
2025-01   5659   3115   8774
2025-02   5460   3580   9040
2025-03   6375   3863  10238
2025-04   6366   2869   9235
2025-05   5992   2779   8771
2025-06   9590   3022  12612
2025-07   8514   2485  10999
2025-08   5915   1846   7761
2025-09   5653   1698   7351
2025-10  10133   2834  12967
2025-11  14025   1971  15996
2025-12  11149   1628  12777
2026-01  11015   2044  13059
2026-02  11151   2338  13489
2026-03   3085    933   4018


In [56]:
# Crear YEAR_WEEK si no existe
for df_base in [recoVTR, recoCLARO]:
    df_base["FECHA_INICIO"] = pd.to_datetime(
        df_base["FECHA_INICIO"],
        errors="coerce"
    )

    iso = df_base["FECHA_INICIO"].dt.isocalendar()
    df_base["ISO_YEAR"] = iso.year
    df_base["ISO_WEEK"] = iso.week

    df_base["YEAR_WEEK"] = (
        df_base["ISO_YEAR"].astype(str)
        + "-W"
        + df_base["ISO_WEEK"].astype(str).str.zfill(2)
    )

In [57]:
# Obtener semanas válidas
semanas = sorted(
    pd.concat([
        recoVTR["YEAR_WEEK"],
        recoCLARO["YEAR_WEEK"]
    ])
    .dropna()
    .unique()
)

ultimas_4 = semanas[-4:]

In [58]:
vtr_4s = recoVTR[recoVTR["YEAR_WEEK"].isin(ultimas_4)]
claro_4s = recoCLARO[recoCLARO["YEAR_WEEK"].isin(ultimas_4)]

vtr_sem = vtr_4s.groupby("YEAR_WEEK").size()
claro_sem = claro_4s.groupby("YEAR_WEEK").size()

conteo_semana = pd.concat([vtr_sem, claro_sem], axis=1)
conteo_semana.columns = ["VTR", "CLARO"]
conteo_semana = conteo_semana.fillna(0).astype(int)

conteo_semana["TOTAL"] = (
    conteo_semana["VTR"] + conteo_semana["CLARO"]
)

print("\n===== CANCELADOS - ÚLTIMAS 4 SEMANAS =====")
print(conteo_semana.sort_index())


===== CANCELADOS - ÚLTIMAS 4 SEMANAS =====
            VTR  CLARO  TOTAL
YEAR_WEEK                    
2026-W08   2234    618   2852
2026-W09   1824    530   2354
2026-W10   2214    691   2905
2026-W11    759    201    960


# Por Logica

In [59]:
recoCLARO.rename(columns={"casillero": "CASILLERO"}, inplace=True)

In [60]:
# Unir ambas bases
base_total = pd.concat([recoVTR, recoCLARO], ignore_index=True)

suma_periodo = (
    base_total
    .groupby("Periodo")[["LOGICA_MASIVO", "LOGICA"]]
    .sum()
    .astype(int)
    .sort_index()
)

print("\n===== SUMA POR PERIODO LOGICAS =====")
print(suma_periodo)


===== SUMA POR PERIODO LOGICAS =====
         LOGICA_MASIVO  LOGICA
Periodo                       
2025-01           3561    5213
2025-02           3424    5616
2025-03           5040    5198
2025-04           4182    5053
2025-05           3610    5161
2025-06           7938    4674
2025-07           5909    5090
2025-08           2768    4993
2025-09           2402    4949
2025-10           5833    7134
2025-11          10444    5552
2025-12           6372    6405
2026-01           5992    7067
2026-02           5440    8049
2026-03           1142    2876


In [61]:
suma_semana = (
    base_total
    .groupby("YEAR_WEEK")[["LOGICA_MASIVO", "LOGICA"]]
    .sum()
    .astype(int)
    .sort_index()
)

# Tomar solo las últimas 5 semanas
suma_semana = suma_semana.tail(30)

print("\n===== SUMA POR SEMANA - ULTIMAS 5 SEMANAS =====")
print(suma_semana)


===== SUMA POR SEMANA - ULTIMAS 5 SEMANAS =====
           LOGICA_MASIVO  LOGICA
YEAR_WEEK                       
2025-W34             836    1282
2025-W35             519    1149
2025-W36             533    1100
2025-W37             522    1173
2025-W38             340    1006
2025-W39             842    1228
2025-W40             665    1505
2025-W41            1274    1519
2025-W42            1312    1635
2025-W43            1470    1635
2025-W44            1857    1572
2025-W45            2970    1539
2025-W46            2575    1272
2025-W47            2356    1174
2025-W48            1963    1277
2025-W49            2003    1390
2025-W50            2264    1540
2025-W51            1378    1500
2025-W52             499    1256
2026-W01             612    1211
2026-W02             850    1429
2026-W03            1232    1694
2026-W04            1303    1630
2026-W05            2473    1939
2026-W06            2344    2130
2026-W07            1474    2121
2026-W08             755   

In [62]:
# Filtrar solo periodo 2026-03
base_filtrada = base_total[base_total["Periodo"] == "2026-03"]

# Asegurar formato fecha
base_filtrada["FECHA_INICIO"] = pd.to_datetime(base_filtrada["FECHA_INICIO"])

# Crear columna DIA
base_filtrada["DIA"] = base_filtrada["FECHA_INICIO"].dt.date

# Suma por día
suma_dia = (
    base_filtrada
    .groupby("DIA")[["LOGICA_MASIVO", "LOGICA"]]
    .sum()
    .astype(int)
    .sort_index()
)

print("\n===== SUMA POR DIA - PERIODO 2026-03 =====")
print(suma_dia)


===== SUMA POR DIA - PERIODO 2026-03 =====
            LOGICA_MASIVO  LOGICA
DIA                              
2026-03-01             44     109
2026-03-02            126     319
2026-03-03            126     356
2026-03-04            112     391
2026-03-05            178     347
2026-03-06            147     307
2026-03-07            112     195
2026-03-08             31     158
2026-03-09            108     346
2026-03-10            135     320
2026-03-11             23      28


C:\Users\wduran\AppData\Local\Temp\ipykernel_15312\2661999351.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base_filtrada["FECHA_INICIO"] = pd.to_datetime(base_filtrada["FECHA_INICIO"])
C:\Users\wduran\AppData\Local\Temp\ipykernel_15312\2661999351.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base_filtrada["DIA"] = base_filtrada["FECHA_INICIO"].dt.date
